In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf
import statsmodels.api as sm

import datetime

In [28]:
pd.set_option('display.max_rows', 200)

In [29]:
df_A_daily = pd.read_excel("data_folder/data.xlsx")
df_A_interval = pd.read_excel("data_folder/data.xlsx", sheet_name="A - Interval")
df_B_daily = pd.read_excel("data_folder/data.xlsx", sheet_name="B - Daily")
df_B_interval = pd.read_excel("data_folder/data.xlsx", sheet_name="B - Interval")
df_C_daily = pd.read_excel("data_folder/data.xlsx", sheet_name="C - Daily")
df_C_interval = pd.read_excel("data_folder/data.xlsx", sheet_name="C - Interval")
df_D_daily = pd.read_excel("data_folder/data.xlsx", sheet_name="D - Daily")
df_D_interval = pd.read_excel("data_folder/data.xlsx", sheet_name="D - Interval") 

df_A_daily_removed = df_A_daily.dropna()
df_B_daily_removed = df_B_daily.dropna()
df_C_daily_removed = df_C_daily.dropna()
df_D_daily_removed = df_D_daily.dropna()
df_A_interval_removed = df_A_interval.dropna()
df_B_interval_removed = df_B_interval.dropna()
df_C_interval_removed = df_C_interval.dropna()
df_D_interval_removed = df_D_interval.dropna()

In [30]:
df_template = pd.read_csv("data_folder/template_forecast_v00.csv")

In [31]:
def convert_HHMM(t):
    return f"{t.hour}:{t.minute:02d}"

def convert_datetime(s):
    hour, min = map(int, s.split(":")) 
    return datetime.time(hour, min)

In [32]:
df_basic_regression = df_template.copy()

In [33]:
# convert call_volume to int (initially float)
df_A_interval_removed['Call Volume'] = df_A_interval_removed['Call Volume'].astype(int)

# poisson regression based on the interval and day
model_a_cv = smf.poisson("Q('Call Volume') ~ C(Interval) + C(Day)", 
                                       data=df_A_interval_removed).fit()

# convert str to datetime 
df_basic_regression['Interval'] = df_basic_regression['Interval'].apply(convert_datetime)

# fill column with predicted values of template df
df_basic_regression["Calls_Offered_A"] = model_a_cv.predict(df_basic_regression).round(0).astype(int)
df_basic_regression


C:\Users\csalg\AppData\Local\Temp\ipykernel_16500\3492057268.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_A_interval_removed['Call Volume'] = df_A_interval_removed['Call Volume'].astype(int)


Optimization terminated successfully.
         Current function value: 8.626167
         Iterations 9


,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,00:00:00,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,August,1,00:30:00,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,August,1,01:00:00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,August,1,01:30:00,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,August,1,02:00:00,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,August,31,21:30:00,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1484,August,31,22:00:00,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,August,31,22:30:00,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1486,August,31,23:00:00,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
df_A_interval_removed["Calls_Offered_A"] = (df_A_interval_removed["Call Volume"] - df_A_interval_removed["Abandoned Calls"]).round(0).astype(int)

C:\Users\csalg\AppData\Local\Temp\ipykernel_16500\1843515894.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_A_interval_removed["Calls_Offered_A"] = (df_A_interval_removed["Call Volume"] - df_A_interval_removed["Abandoned Calls"]).round(0).astype(int)


In [35]:
model_a_cct = smf.ols("Q('CCT') ~ C(Interval) + C(Day) + Q('Calls_Offered_A')", 
                      data=df_A_interval_removed).fit()

df_basic_regression["CCT_A"] = model_a_cct.predict(df_basic_regression)



model_a_abd = smf.glm("Q('Abandoned Rate') ~ C(Interval) + C(Day)",
                        data=df_A_interval_removed, family = sm.families.Binomial()).fit()

df_basic_regression["Abandoned_Rate_A"] = model_a_abd.predict(df_basic_regression)

df_basic_regression["Abandoned_Calls_A"] = df_basic_regression["Calls_Offered_A"] * df_basic_regression["Abandoned_Rate_A"]
df_basic_regression["Abandoned_Calls_A"] = (df_basic_regression["Abandoned_Calls_A"].round(0).astype(int))

df_basic_regression

,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,00:00:00,5,0,0.037960,374.401851,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,August,1,00:30:00,4,0,0.049922,318.839086,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,August,1,01:00:00,3,0,0.048953,351.805866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,August,1,01:30:00,2,0,0.058645,305.395604,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,August,1,02:00:00,2,0,0.018942,353.503453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,August,31,21:30:00,17,0,0.026455,293.221570,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1484,August,31,22:00:00,14,0,0.010960,295.883183,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,August,31,22:30:00,10,2,0.202786,288.267712,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1486,August,31,23:00:00,8,1,0.106866,306.241289,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
df_B_interval_removed['Call Volume'] = df_B_interval_removed['Call Volume'].astype(int)
model_b_cv = smf.poisson("Q('Call Volume') ~ C(Interval) + C(Day)", 
                                       data=df_B_interval_removed).fit()
df_basic_regression["Calls_Offered_B"] = model_b_cv.predict(df_basic_regression).round(0).astype(int)


df_B_interval_removed["Calls_Offered_B"] = (df_B_interval_removed["Call Volume"] - df_B_interval_removed["Abandoned Calls"]).round(0).astype(int)


model_b_cct = smf.ols("Q('CCT') ~ C(Interval) + C(Day) + Q('Calls_Offered_B')", 
                      data=df_B_interval_removed).fit()

df_basic_regression["CCT_B"] = model_b_cct.predict(df_basic_regression)


model_b_abd = smf.glm("Q('Abandoned Rate') ~ C(Interval) + C(Day)",
                        data=df_B_interval_removed, family = sm.families.Binomial()).fit()

df_basic_regression["Abandoned_Rate_B"] = model_a_abd.predict(df_basic_regression)

df_basic_regression["Abandoned_Calls_B"] = df_basic_regression["Calls_Offered_B"] * df_basic_regression["Abandoned_Rate_B"]
df_basic_regression["Abandoned_Calls_B"] = (df_basic_regression["Abandoned_Calls_B"].round(0).astype(int))

df_basic_regression


C:\Users\csalg\AppData\Local\Temp\ipykernel_16500\2289357089.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_B_interval_removed['Call Volume'] = df_B_interval_removed['Call Volume'].astype(int)


Optimization terminated successfully.
         Current function value: 9.854434
         Iterations 10


C:\Users\csalg\AppData\Local\Temp\ipykernel_16500\2289357089.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_B_interval_removed["Calls_Offered_B"] = (df_B_interval_removed["Call Volume"] - df_B_interval_removed["Abandoned Calls"]).round(0).astype(int)


,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,00:00:00,5,0,0.037960,374.401851,29,1,0.037960,316.673094,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,August,1,00:30:00,4,0,0.049922,318.839086,21,1,0.049922,304.212395,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,August,1,01:00:00,3,0,0.048953,351.805866,9,0,0.048953,362.026234,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,August,1,01:30:00,2,0,0.058645,305.395604,5,0,0.058645,327.892681,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,August,1,02:00:00,2,0,0.018942,353.503453,4,0,0.018942,340.320076,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,August,31,21:30:00,17,0,0.026455,293.221570,92,2,0.026455,296.875165,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1484,August,31,22:00:00,14,0,0.010960,295.883183,74,1,0.010960,290.872913,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,August,31,22:30:00,10,2,0.202786,288.267712,58,12,0.202786,283.870047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1486,August,31,23:00:00,8,1,0.106866,306.241289,43,5,0.106866,275.518144,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
df_C_interval_removed['Call Volume'] = df_C_interval_removed['Call Volume'].astype(int)
model_c_cv = smf.poisson("Q('Call Volume') ~ C(Interval) + C(Day)", 
                                       data=df_C_interval_removed).fit()
df_basic_regression["Calls_Offered_C"] = model_c_cv.predict(df_basic_regression).round(0).astype(int)


df_C_interval_removed["Calls_Offered_C"] = (df_C_interval_removed["Call Volume"] - df_C_interval_removed["Abandoned Calls"]).round(0).astype(int)


model_c_cct = smf.ols("Q('CCT') ~ C(Interval) + C(Day) + Q('Calls_Offered_C')", 
                      data=df_C_interval_removed).fit()

df_basic_regression["CCT_C"] = model_c_cct.predict(df_basic_regression)


model_c_abd = smf.glm("Q('Abandoned Rate') ~ C(Interval) + C(Day)",
                        data=df_C_interval_removed, family = sm.families.Binomial()).fit()

df_basic_regression["Abandoned_Rate_C"] = model_c_abd.predict(df_basic_regression)

df_basic_regression["Abandoned_Calls_C"] = df_basic_regression["Calls_Offered_C"] * df_basic_regression["Abandoned_Rate_C"]
df_basic_regression["Abandoned_Calls_C"] = (df_basic_regression["Abandoned_Calls_C"].round(0).astype(int))

df_basic_regression

C:\Users\csalg\AppData\Local\Temp\ipykernel_16500\903377259.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_C_interval_removed['Call Volume'] = df_C_interval_removed['Call Volume'].astype(int)


Optimization terminated successfully.
         Current function value: 18.689567
         Iterations 9


C:\Users\csalg\AppData\Local\Temp\ipykernel_16500\903377259.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_C_interval_removed["Calls_Offered_C"] = (df_C_interval_removed["Call Volume"] - df_C_interval_removed["Abandoned Calls"]).round(0).astype(int)


,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,00:00:00,5,0,0.037960,374.401851,29,1,0.037960,316.673094,74,0,0.003586,317.109143,NaN,NaN,NaN,NaN
1,August,1,00:30:00,4,0,0.049922,318.839086,21,1,0.049922,304.212395,56,0,0.002049,312.736980,NaN,NaN,NaN,NaN
2,August,1,01:00:00,3,0,0.048953,351.805866,9,0,0.048953,362.026234,41,0,0.005339,308.837304,NaN,NaN,NaN,NaN
3,August,1,01:30:00,2,0,0.058645,305.395604,5,0,0.058645,327.892681,31,0,0.006165,310.596437,NaN,NaN,NaN,NaN
4,August,1,02:00:00,2,0,0.018942,353.503453,4,0,0.018942,340.320076,21,0,0.015179,287.451636,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,August,31,21:30:00,17,0,0.026455,293.221570,92,2,0.026455,296.875165,177,1,0.003894,334.262063,NaN,NaN,NaN,NaN
1484,August,31,22:00:00,14,0,0.010960,295.883183,74,1,0.010960,290.872913,146,1,0.006608,330.908256,NaN,NaN,NaN,NaN
1485,August,31,22:30:00,10,2,0.202786,288.267712,58,12,0.202786,283.870047,116,1,0.004554,324.074551,NaN,NaN,NaN,NaN
1486,August,31,23:00:00,8,1,0.106866,306.241289,43,5,0.106866,275.518144,90,0,0.002795,321.684753,NaN,NaN,NaN,NaN


In [38]:
df_D_interval_removed['Call Volume'] = df_D_interval_removed['Call Volume'].astype(int)
model_d_cv = smf.poisson("Q('Call Volume') ~ C(Interval) + C(Day)", 
                                       data=df_D_interval_removed).fit()
df_basic_regression["Calls_Offered_D"] = model_d_cv.predict(df_basic_regression).round(0).astype(int)


df_D_interval_removed["Calls_Offered_D"] = (df_D_interval_removed["Call Volume"] - df_D_interval_removed["Abandoned Calls"]).round(0).astype(int)


model_d_cct = smf.ols("Q('CCT') ~ C(Interval) + C(Day) + Q('Calls_Offered_D')", 
                      data=df_D_interval_removed).fit()

df_basic_regression["CCT_D"] = model_d_cct.predict(df_basic_regression)


model_d_abd = smf.glm("Q('Abandoned Rate') ~ C(Interval) + C(Day)",
                        data=df_D_interval_removed, family = sm.families.Binomial()).fit()

df_basic_regression["Abandoned_Rate_D"] = model_d_abd.predict(df_basic_regression)

df_basic_regression["Abandoned_Calls_D"] = df_basic_regression["Calls_Offered_D"] * df_basic_regression["Abandoned_Rate_D"]
df_basic_regression["Abandoned_Calls_D"] = (df_basic_regression["Abandoned_Calls_D"].round(0).astype(int))

df_basic_regression

C:\Users\csalg\AppData\Local\Temp\ipykernel_16500\392329454.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_D_interval_removed['Call Volume'] = df_D_interval_removed['Call Volume'].astype(int)


Optimization terminated successfully.
         Current function value: 12.309625
         Iterations 10


C:\Users\csalg\AppData\Local\Temp\ipykernel_16500\392329454.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_D_interval_removed["Calls_Offered_D"] = (df_D_interval_removed["Call Volume"] - df_D_interval_removed["Abandoned Calls"]).round(0).astype(int)


,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,00:00:00,5,0,0.037960,374.401851,29,1,0.037960,316.673094,74,0,0.003586,317.109143,38,0,0.003234,310.639393
1,August,1,00:30:00,4,0,0.049922,318.839086,21,1,0.049922,304.212395,56,0,0.002049,312.736980,27,0,0.001183,322.725210
2,August,1,01:00:00,3,0,0.048953,351.805866,9,0,0.048953,362.026234,41,0,0.005339,308.837304,20,0,0.002164,337.674914
3,August,1,01:30:00,2,0,0.058645,305.395604,5,0,0.058645,327.892681,31,0,0.006165,310.596437,16,0,0.004829,329.880067
4,August,1,02:00:00,2,0,0.018942,353.503453,4,0,0.018942,340.320076,21,0,0.015179,287.451636,11,0,0.012634,331.614853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,August,31,21:30:00,17,0,0.026455,293.221570,92,2,0.026455,296.875165,177,1,0.003894,334.262063,100,1,0.013767,291.860799
1484,August,31,22:00:00,14,0,0.010960,295.883183,74,1,0.010960,290.872913,146,1,0.006608,330.908256,82,1,0.011528,292.449767
1485,August,31,22:30:00,10,2,0.202786,288.267712,58,12,0.202786,283.870047,116,1,0.004554,324.074551,68,1,0.007865,288.781087
1486,August,31,23:00:00,8,1,0.106866,306.241289,43,5,0.106866,275.518144,90,0,0.002795,321.684753,53,0,0.004426,289.481406


In [39]:
df_basic_regression["Abandoned_Rate_A"] = (df_basic_regression["Abandoned_Rate_A"] * 100).round(2).astype(str) + "%"
df_basic_regression["Abandoned_Rate_B"] = (df_basic_regression["Abandoned_Rate_B"] * 100).round(2).astype(str) + "%"
df_basic_regression["Abandoned_Rate_C"] = (df_basic_regression["Abandoned_Rate_C"] * 100).round(2).astype(str) + "%"
df_basic_regression["Abandoned_Rate_D"] = (df_basic_regression["Abandoned_Rate_D"] * 100).round(2).astype(str) + "%"

In [40]:
df_basic_regression

,Month,Day,Interval,Calls_Offered_A,Abandoned_Calls_A,Abandoned_Rate_A,CCT_A,Calls_Offered_B,Abandoned_Calls_B,Abandoned_Rate_B,CCT_B,Calls_Offered_C,Abandoned_Calls_C,Abandoned_Rate_C,CCT_C,Calls_Offered_D,Abandoned_Calls_D,Abandoned_Rate_D,CCT_D
0,August,1,00:00:00,5,0,3.8%,374.401851,29,1,3.8%,316.673094,74,0,0.36%,317.109143,38,0,0.32%,310.639393
1,August,1,00:30:00,4,0,4.99%,318.839086,21,1,4.99%,304.212395,56,0,0.2%,312.736980,27,0,0.12%,322.725210
2,August,1,01:00:00,3,0,4.9%,351.805866,9,0,4.9%,362.026234,41,0,0.53%,308.837304,20,0,0.22%,337.674914
3,August,1,01:30:00,2,0,5.86%,305.395604,5,0,5.86%,327.892681,31,0,0.62%,310.596437,16,0,0.48%,329.880067
4,August,1,02:00:00,2,0,1.89%,353.503453,4,0,1.89%,340.320076,21,0,1.52%,287.451636,11,0,1.26%,331.614853
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1483,August,31,21:30:00,17,0,2.65%,293.221570,92,2,2.65%,296.875165,177,1,0.39%,334.262063,100,1,1.38%,291.860799
1484,August,31,22:00:00,14,0,1.1%,295.883183,74,1,1.1%,290.872913,146,1,0.66%,330.908256,82,1,1.15%,292.449767
1485,August,31,22:30:00,10,2,20.28%,288.267712,58,12,20.28%,283.870047,116,1,0.46%,324.074551,68,1,0.79%,288.781087
1486,August,31,23:00:00,8,1,10.69%,306.241289,43,5,10.69%,275.518144,90,0,0.28%,321.684753,53,0,0.44%,289.481406


In [41]:
df_basic_regression.to_csv("forecast_v02.csv", index = False)